In [1]:
from pathlib import Path
import pandas as pd
from ib_async import IB, Stock, MarketOrder

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Market Data" / "Cleaned"
ORDERS_PARQUET = DATA_DIR / "paper_orders_only.parquet"

orders = pd.read_parquet(ORDERS_PARQUET)
orders["date"] = pd.to_datetime(orders["date"])

display(orders)

,date,symbol,bucket,close,target_weight,target_dollars,target_shares,current_shares,delta_shares,side,estimated_trade_dollars,abs_delta_shares
0,2026-03-06,SPY,core_us,672.38,0.300000,300025.905000,446,0,446,BUY,299881.48,446
1,2026-03-06,QQQ,core_us,599.75,0.300000,300025.905000,500,0,500,BUY,299875.00,500
2,2026-03-06,VPU,us_sector,201.60,0.050000,50004.317500,248,0,248,BUY,49996.80,248
3,2026-03-06,VFH,us_sector,123.43,0.050000,50004.317500,405,0,405,BUY,49989.15,405
4,2026-03-06,VHT,us_sector,282.15,0.050000,50004.317500,177,0,177,BUY,49940.55,177
5,2026-03-06,VIS,us_sector,326.26,0.050000,50004.317500,153,0,153,BUY,49917.78,153
6,2026-03-06,VWO,international,54.47,0.033333,33336.211667,612,0,612,BUY,33335.64,612
7,2026-03-06,INDA,international,49.99,0.033333,33336.211667,666,0,666,BUY,33293.34,666
8,2026-03-06,VEA,international,65.28,0.033333,33336.211667,510,0,510,BUY,33292.80,510
9,2026-03-06,SHY,defensive,82.73,0.012500,12501.079375,151,0,151,BUY,12492.23,151


In [3]:
SUBMIT_ORDERS = False
TEST_SYMBOLS = ["SPY", "QQQ"]   # Monday: start with just these
CLIENT_ID = 11
HOST = "127.0.0.1"
PORT = 4004

In [4]:
ib = IB()
await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)

print("Connected:", ib.isConnected())
print("Managed accounts:", ib.managedAccounts())

Connected: True
Managed accounts: ['DUP317554']


In [5]:
orders_to_send = orders[orders["symbol"].isin(TEST_SYMBOLS)].copy().reset_index(drop=True)

display(
    orders_to_send[
        ["symbol", "side", "delta_shares", "estimated_trade_dollars"]
    ]
)

,symbol,side,delta_shares,estimated_trade_dollars
0,SPY,BUY,446,299881.48
1,QQQ,BUY,500,299875.00


In [6]:
contracts = [Stock(symbol, "SMART", "USD") for symbol in orders_to_send["symbol"]]

qualified = await ib.qualifyContractsAsync(*contracts)

print("Qualified contracts:")
for c in qualified:
    print(c)

Qualified contracts:
Stock(conId=756733, symbol='SPY', exchange='SMART', primaryExchange='ARCA', currency='USD', localSymbol='SPY', tradingClass='SPY')
Stock(conId=320227571, symbol='QQQ', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='QQQ', tradingClass='NMS')


In [7]:
trade_plan = []

for _, row in orders_to_send.iterrows():
    symbol = row["symbol"]
    side = row["side"]
    qty = int(abs(row["delta_shares"]))

    contract = next(c for c in qualified if c.symbol == symbol)

    trade_plan.append(
        {
            "symbol": symbol,
            "side": side,
            "qty": qty,
            "contract": contract,
        }
    )

pd.DataFrame([{k: v for k, v in x.items() if k != "contract"} for x in trade_plan])

,symbol,side,qty
0,SPY,BUY,446
1,QQQ,BUY,500


In [8]:
for item in trade_plan:
    print(f"{item['side']} {item['qty']} shares of {item['symbol']}")

BUY 446 shares of SPY
BUY 500 shares of QQQ


In [9]:
submitted_trades = []

if SUBMIT_ORDERS:
    for item in trade_plan:
        order = MarketOrder(item["side"], item["qty"])
        trade = ib.placeOrder(item["contract"], order)
        submitted_trades.append(trade)
        print(f"Submitted: {item['side']} {item['qty']} {item['symbol']}")
else:
    print("Dry run only. No orders submitted.")

Dry run only. No orders submitted.


In [10]:
if SUBMIT_ORDERS:
    await ib.sleepAsync(5)

    for trade in submitted_trades:
        print("-----")
        print("Symbol:", trade.contract.symbol)
        print("Status:", trade.orderStatus.status)
        print("Filled:", trade.orderStatus.filled)
        print("Remaining:", trade.orderStatus.remaining)
        print("Trade log:")
        for entry in trade.log:
            print(entry)

In [11]:
positions = ib.positions()

positions_df = pd.DataFrame(
    [
        {
            "symbol": getattr(p.contract, "symbol", None),
            "shares": float(p.position),
            "avg_cost": float(p.avgCost),
        }
        for p in positions
    ]
)

display(positions_df)

""


In [12]:
ib.disconnect()
print("Disconnected.")

Disconnected.
